# NeuroClass AI Agent Training
Connect to Supabase, fetch student queries & course content, build a RAG vector store,
and train a LangGraph tutor agent — fully runnable in **Google Colab**.

**Setup:** In Colab go to *Runtime → Manage secrets* and add:
- `SUPABASE_URL`
- `SUPABASE_SERVICE_ROLE_KEY`
- `GEMINI_API_KEY`


In [ ]:
# Cell 1 — Install dependencies
!pip install -q supabase langchain langchain-google-genai langchain-community langgraph pypdf pandas faiss-cpu

In [ ]:
# Cell 2 — Load credentials (Colab Secrets preferred)
import os

try:
    from google.colab import userdata
    SUPABASE_URL             = userdata.get('SUPABASE_URL')
    SUPABASE_SERVICE_ROLE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
    GEMINI_API_KEY           = userdata.get('GEMINI_API_KEY')
except Exception:
    SUPABASE_URL             = os.getenv('SUPABASE_URL', 'https://tvizwaysproajwebglwv.supabase.co')
    SUPABASE_SERVICE_ROLE_KEY = os.getenv('SUPABASE_SERVICE_ROLE_KEY', '')
    GEMINI_API_KEY           = os.getenv('GEMINI_API_KEY', '')

os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY
print('Credentials loaded.')

In [ ]:
# Cell 3 — Connect to Supabase (service_role bypasses RLS for training)
from supabase import create_client
import pandas as pd

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print('Connected to Supabase:', SUPABASE_URL)

In [ ]:
# Cell 4 — Fetch student queries (primary AI training data)
resp = sb.table('student_queries').select('*').order('created_at').execute()
queries_df = pd.DataFrame(resp.data)
print(f'Loaded {len(queries_df)} student queries')
if not queries_df.empty:
    print(queries_df[['created_at','query_text','provider']].tail(5).to_string())

In [ ]:
# Cell 5 — Fetch uploaded file metadata
resp = sb.table('uploaded_files').select('id,course_id,file_name,file_type,storage_bucket,storage_path,created_at').execute()
files_df = pd.DataFrame(resp.data)
print(f'Loaded {len(files_df)} file records')
if not files_df.empty:
    print(files_df.groupby('file_type')['id'].count().rename('count').to_string())

In [ ]:
# Cell 6 — Download a PDF from Supabase Storage and extract text
import io
from pypdf import PdfReader
from langchain_core.documents import Document

def load_pdf_from_supabase(row):
    try:
        raw = sb.storage.from_(row['storage_bucket']).download(row['storage_path'])
        reader = PdfReader(io.BytesIO(raw))
        text = '\n'.join(p.extract_text() or '' for p in reader.pages)
        return Document(page_content=text, metadata={'source': row['file_name'], 'course_id': row['course_id']})
    except Exception as e:
        print(f"  WARNING: Could not load {row['file_name']}: {e}")
        return None

pdf_rows = files_df[files_df['file_type'] == 'pdf'].to_dict('records') if not files_df.empty else []
raw_docs = [load_pdf_from_supabase(r) for r in pdf_rows[:10]]  # limit to 10 for demo
docs = [d for d in raw_docs if d]
print(f'Loaded {len(docs)} PDF documents from Supabase Storage')

In [ ]:
# Cell 7 — Also load chunked text from course_contents table
resp = sb.table('course_contents').select('id,course_id,content_chunk').execute()
contents_df = pd.DataFrame(resp.data)
print(f'Loaded {len(contents_df)} course content chunks from DB')

db_docs = [
    Document(page_content=row['content_chunk'], metadata={'course_id': row['course_id']})
    for _, row in contents_df.iterrows()
    if row.get('content_chunk')
]
all_docs = docs + db_docs
print(f'Total documents for RAG: {len(all_docs)}')

In [ ]:
# Cell 8 — Build FAISS vector store with Gemini embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

embeddings = GoogleGenerativeAIEmbeddings(model='models/embedding-001')

if all_docs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_documents(all_docs)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
    print(f'FAISS vectorstore built: {len(chunks)} chunks')
else:
    retriever = None
    print('No documents yet. Upload course materials in the NeuroClass app first.')

In [ ]:
# Cell 9 — LangGraph Tutor Agent (RAG + Supabase query logging)
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, List, Optional, Any
import uuid

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.3)

class TutorState(TypedDict):
    messages: List[Any]
    course_context: str
    student_id: Optional[str]
    course_id: Optional[str]
    thread_id: str
    db_query_id: Optional[str]

def save_query_node(state: TutorState) -> TutorState:
    last = next((m.content for m in reversed(state['messages']) if isinstance(m, HumanMessage)), '')
    try:
        result = sb.table('student_queries').insert({
            'student_id': state.get('student_id'),
            'course_id':  state.get('course_id'),
            'thread_id':  state['thread_id'],
            'query_text': last,
            'provider':   'gemini',
        }).execute()
        state['db_query_id'] = result.data[0]['id']
    except Exception as e:
        print('  WARNING: could not save query:', e)
    return state

def retrieve_context_node(state: TutorState) -> TutorState:
    if retriever is None:
        return {**state, 'course_context': ''}
    last = next((m.content for m in reversed(state['messages']) if isinstance(m, HumanMessage)), '')
    ctx_docs = retriever.invoke(last)
    ctx = '\n\n'.join(d.page_content for d in ctx_docs)
    return {**state, 'course_context': ctx}

def answer_node(state: TutorState) -> TutorState:
    system = (
        'You are a helpful AI tutor for NeuroClass. Guide students with Socratic questioning.\n'
        f"Relevant course material:\n{state['course_context']}"
    )
    msgs = [HumanMessage(content=system)] + state['messages']
    reply = llm.invoke(msgs)
    return {**state, 'messages': state['messages'] + [reply]}

def save_response_node(state: TutorState) -> TutorState:
    if state.get('db_query_id'):
        last_ai = next((m.content for m in reversed(state['messages']) if isinstance(m, AIMessage)), '')
        try:
            sb.table('student_queries').update({'ai_reply': last_ai}).eq('id', state['db_query_id']).execute()
        except Exception as e:
            print('  WARNING: could not save response:', e)
    return state

graph = StateGraph(TutorState)
graph.add_node('save_query',      save_query_node)
graph.add_node('retrieve_context', retrieve_context_node)
graph.add_node('answer',          answer_node)
graph.add_node('save_response',   save_response_node)
graph.set_entry_point('save_query')
graph.add_edge('save_query',       'retrieve_context')
graph.add_edge('retrieve_context', 'answer')
graph.add_edge('answer',           'save_response')
graph.add_edge('save_response',    END)
tutor_agent = graph.compile()
print('LangGraph tutor agent compiled.')

In [ ]:
# Cell 10 — Run the agent (test query)
result = tutor_agent.invoke({
    'messages':      [HumanMessage(content='What is gradient descent?')],
    'course_context': '',
    'student_id':    None,   # replace with real UUID from profiles table
    'course_id':     None,   # replace with real course UUID
    'thread_id':     str(uuid.uuid4()),
    'db_query_id':   None,
})
print('Agent reply:', result['messages'][-1].content)

In [ ]:
# Cell 11 — Analyse student query patterns
if not queries_df.empty:
    print('Top 10 recent queries:')
    print(queries_df[['created_at','query_text','provider']].tail(10).to_string())
    if 'course_id' in queries_df.columns:
        print('\nQuery count by course:')
        print(queries_df.groupby('course_id')['id'].count().rename('queries'))
else:
    print('No student queries yet.')

In [ ]:
# Cell 12 — Export fine-tuning dataset (input/output pairs) to CSV
if not queries_df.empty and 'ai_reply' in queries_df.columns:
    dataset = queries_df[['query_text','ai_reply','course_id','created_at']].dropna(subset=['query_text','ai_reply'])
    dataset = dataset.rename(columns={'query_text':'input','ai_reply':'output'})
    dataset.to_csv('neuroclass_training_dataset.csv', index=False)
    print(f'Saved {len(dataset)} rows → neuroclass_training_dataset.csv')
else:
    print('No labelled data to export yet.')